# 🎬 Inferencia integrada — Detección + Segmentación + Pose (YOLO26)

Pipeline que aplica **3 modelos en paralelo** a cada frame de los videos de `videos/input/`:
1. **Detección custom** (`best.pt`): factura, mate, termo
2. **Segmentación COCO** (`yolo26n-seg.pt`), excluyendo `person` y las clases custom
3. **Pose humana** (`yolo26n-pose.pt`), skeleton COCO-17

Los videos anotados se guardan en `videos/output/`.

## 0. ⚙️ Entorno de ejecución (Colab o local)

Esta celda funciona en ambos entornos: en **Colab** monta Google Drive y se ubica en la carpeta del repo; en **local** busca la raíz del repo automáticamente.

In [ ]:
import os
from pathlib import Path

try:
    # En Google Colab: montar Drive y ubicarse en la carpeta del repo
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/UGR/PDI_TP2/TPF---Vision-por-Computadora-2026')
    print("Entorno: Google Colab")
except ImportError:
    # En local: buscar la raíz del repo (la carpeta que contiene data/mate-termo)
    cwd = Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'data' / 'mate-termo').exists():
            os.chdir(candidate)
            break
    print("Entorno: local")

print(f"Directorio de trabajo: {Path.cwd()}")

## 1. 📦 Instalación de dependencias

In [ ]:
# -U asegura una version de ultralytics reciente (YOLO26 requiere >= enero 2026)
!pip install -q -U ultralytics opencv-python numpy torch torchvision Pillow matplotlib

## 2. 📚 Imports y configuración

Importamos las librerías y agregamos `inference/` al path para usar `utils.py`.

In [ ]:
import sys
import os
import time
from pathlib import Path

# Agregar inference/ al path para importar utils
sys.path.insert(0, 'inference')

import cv2
import numpy as np
from ultralytics import YOLO

import utils

print("✅ Librerías importadas correctamente")

## 3. 🧠 Cargar los 3 modelos

Se cargan los 3 modelos en paralelo (los archivos `.pt` se descargan automáticamente la primera vez).

> ℹ️ `yolo26n-seg.pt` y `yolo26n-pose.pt` existen en `ultralytics` desde enero 2026 (la celda 1 actualiza la librería). `best.pt` es el resultado del notebook de entrenamiento — correrlo primero.

In [ ]:
# Ajustar los nombres de los modelos segun la version disponible
SEG_MODEL_NAME    = 'yolo26n-seg.pt'    # COCO segmentation
POSE_MODEL_NAME   = 'yolo26n-pose.pt'   # pose estimation
CUSTOM_MODEL_PATH = 'best.pt'           # detector custom (entregable del training)

print("Cargando modelos...")
seg_model    = YOLO(SEG_MODEL_NAME)
pose_model   = YOLO(POSE_MODEL_NAME)
custom_model = YOLO(CUSTOM_MODEL_PATH)
print("✅ 3 modelos cargados:")
print(f"  - Segmentacion:  {SEG_MODEL_NAME}")
print(f"  - Pose:          {POSE_MODEL_NAME}")
print(f"  - Custom:        {CUSTOM_MODEL_PATH} ({list(custom_model.names.values())})")

## 4. 🎥 Configurar videos de entrada y salida

Se procesan **todos** los videos de `videos/input/` (la consigna pide 2-3 videos de ~20s). Cada `videoN.mp4` genera un `videos/output/resultado_videoN.mp4`.

In [ ]:
from pathlib import Path

VIDEO_DIR = Path('videos/input')
OUTPUT_DIR = Path('videos/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

videos = sorted(VIDEO_DIR.glob('*.mp4')) + sorted(VIDEO_DIR.glob('*.avi'))
if not videos:
    raise FileNotFoundError(f"❌ No hay videos en {VIDEO_DIR.resolve()} — copiarlos a videos/input/")

print(f"Se van a procesar {len(videos)} videos:")
for v in videos:
    cap = cv2.VideoCapture(str(v))
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    dur = n / fps if fps else 0
    print(f"  - {v.name}: {w}x{h} @ {fps:.1f} FPS, {n} frames ({dur:.1f}s)")

## 5. 🎞️ Procesar todos los videos frame a frame

Para cada video: se aplican los 3 modelos a cada frame, se combinan los resultados en un único frame anotado (con overlay de FPS y conteos) y se escribe el video de salida.

> 💡 Para un test rápido, poner `MAX_FRAMES = 100` (procesa solo los primeros 100 frames de cada video).

In [ ]:
CONFIDENCE = 0.4
CUSTOM_CLASSES = list(custom_model.names.values())  # ['factura', 'mate', 'termo']
MAX_FRAMES = None  # int para procesar solo N frames por video (debug), None para todo


def procesar_video(video_path, output_path):
    """Aplica los 3 modelos frame a frame y escribe el video anotado."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"❌ No se pudo abrir {video_path}")

    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    frame_count = 0
    t_start = time.time()
    try:
        while True:
            ok, frame = cap.read()
            if not ok or (MAX_FRAMES is not None and frame_count >= MAX_FRAMES):
                break
            frame_count += 1

            # 1) Segmentacion COCO (excluye 'person' y clases custom)
            seg_res = seg_model(frame, conf=CONFIDENCE, verbose=False)[0]

            # 2) Pose estimation (personas)
            pose_res = pose_model(frame, conf=CONFIDENCE, verbose=False)[0]

            # 3) Deteccion de clases custom
            cust_res = custom_model(frame, conf=CONFIDENCE, verbose=False)[0]

            # 4) Combinar todo en un unico frame
            annotated, counts = utils.annotate_frame(
                frame=frame,
                seg_result=seg_res,
                pose_result=pose_res,
                custom_result=cust_res,
                seg_model_names=seg_model.names,
                pose_model_names=pose_model.names,
                custom_model_names=custom_model.names,
                custom_class_names=CUSTOM_CLASSES,
                conf_threshold=CONFIDENCE,
            )

            # 5) Calcular FPS y overlay
            elapsed = time.time() - t_start
            current_fps = frame_count / elapsed if elapsed > 0 else 0
            annotated = utils.add_info_overlay(
                annotated, current_fps,
                custom_count=counts['n_custom'],
                seg_count=counts['n_seg'],
                person_count=counts['n_persons'],
            )

            out.write(annotated)

            if frame_count % 30 == 0:
                print(f"    frame {frame_count}/{total_frames} - FPS: {current_fps:.1f}")
    finally:
        cap.release()
        out.release()

    elapsed = time.time() - t_start
    print(f"  ✅ {frame_count} frames en {elapsed:.1f}s ({frame_count / max(elapsed, 1e-6):.1f} FPS promedio)")
    print(f"     Guardado en: {output_path}")


output_paths = []
for v in videos:
    out_path = OUTPUT_DIR / f"resultado_{v.stem}.mp4"
    print(f"\n🎥 Procesando {v.name} ...")
    procesar_video(v, out_path)
    output_paths.append(out_path)

print(f"\n✅ {len(output_paths)} videos procesados en {OUTPUT_DIR}/")

## 6. 📊 Verificar los resultados

Muestreo de 3 frames (inicio / medio / fin) de cada video procesado para verificar visualmente la salida. Estos frames también sirven como material para las slides 7, 10 y 11 de la presentación.

In [ ]:
import matplotlib.pyplot as plt

for out_path in output_paths:
    out_cap = cv2.VideoCapture(str(out_path))
    n = int(out_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    samples = [0, n // 2, n - 1] if n >= 3 else list(range(n))

    fig, axes = plt.subplots(1, max(len(samples), 1), figsize=(18, 5))
    if len(samples) <= 1:
        axes = [axes]

    for ax, idx in zip(axes, samples):
        out_cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = out_cap.read()
        if ok:
            ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{out_path.name} — frame {idx}", fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    out_cap.release()

print("Videos completos en:", *[str(p) for p in output_paths], sep='\n  ')